# Week 3 Guided Lab: First Steps in Data Cleaning

**BAN 6003: Data Management and Analytics Integration**

In Week 2, you acted as a data detective. You loaded a dataset, inspected its structure, checked missing values, reviewed summary statistics, and looked for duplicates.

This week, you start acting as a careful data surgeon. Your job is not to randomly change the dataset. Your job is to make deliberate, documented cleaning decisions.

In this guided lab, we will use `employee_data.csv` to practice a first cleaning pass.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Make a safe working copy of a raw DataFrame.
2. Correct common data type problems using `pd.to_numeric()` and `pd.to_datetime()`.
3. Understand what `errors="coerce"` does.
4. Identify full-row duplicates and key-based duplicates.
5. Remove duplicate employee IDs when `employee_id` should be unique.
6. Compare missing-data strategies: dropping and imputation.
7. Impute missing numeric values using the median.
8. Drop rows with missing critical identifiers or dates.
9. Verify the cleaned dataset using `info()`, `shape`, and `isna().sum()`.
10. Keep a simple cleaning log that documents your decisions.

## Important: Cleaning Is Not Guessing

In practice, we need to be careful. A cleaning decision should be based on the business meaning of the column.

For example, if `employee_id` is the unique identifier for each employee, then a duplicate employee ID is a serious issue. If `salary` is missing, we may be able to impute it in a classroom exercise, but in a real HR or financial services environment we might need to investigate the source system before filling it in.

This lab teaches the mechanics. Your project will require you to explain the reasoning.

## Part 1: Import Pandas and Load the Raw Data

We will start by importing Pandas and loading the same type of employee dataset used in the Week 2 profiling activity.

In [3]:
from pathlib import Path
import pandas as pd

# Load the protected Week 3 raw employee data.
raw_path = Path("/workspaces/ban6003-week1-2-pip/data/employee_data.csv")
df = pd.read_csv(raw_path)

df.head()

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
0,101,John,Smith,4/15/22,Marketing,85000,34.0,True
1,102,Jane,Doe,8/20/21,Sales,92000,45.0,True
2,103,Peter,Jones,1/2/22,HR,71000,29.0,True
3,104,Mary,Johnson,11/10/23,Engineering,115000,38.0,True
4,105,Mike,Brown,5/30/20,sales,89000,NaN,False


## Part 2: Review the Raw Data

Before cleaning anything, inspect the dataset again. We want to confirm the shape, data types, missing values, and suspicious values.

In [4]:
# How many rows and columns?
df.shape

(13, 8)

In [5]:
# What data types did Pandas assign?
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   employee_id  13 non-null     str    
 1   first_name   13 non-null     str    
 2   last_name    13 non-null     str    
 3   hire_date    12 non-null     str    
 4   department   13 non-null     str    
 5   salary       12 non-null     str    
 6   age          12 non-null     float64
 7   full_time    12 non-null     object 
dtypes: float64(1), object(1), str(6)
memory usage: 964.0+ bytes


In [6]:
# Count missing values by column
df.isna().sum()

employee_id    0
first_name     0
last_name      0
hire_date      1
department     0
salary         1
age            1
full_time      1
dtype: int64

In [7]:
# First look at all rows for this small demo dataset
df

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
0,101,John,Smith,4/15/22,Marketing,85000,34.0,True
1,102,Jane,Doe,8/20/21,Sales,92000,45.0,True
2,103,Peter,Jones,1/2/22,HR,71000,29.0,True
3,104,Mary,Johnson,11/10/23,Engineering,115000,38.0,True
4,105,Mike,Brown,5/30/20,sales,89000,NaN,False
5,106,Susan,Davis,NaN,Sales,95000,41.0,True
6,107,Chris,Wilson,2023-FEB-21,Marketing,NaN,33.0,True
7,108,Laura,Taylor,9/1/21,Engineering,120000,52.0,NaN
8,102,Emily,White,1/15/24,Sales,91000,31.0,True
9,110,David,Miller,7/22/19,HR,68000,58.0,True


### Think Before Cleaning

Look at the output above. You should notice several issues:

- `employee_id` should be numeric, but one row contains the text value `error`.
- `hire_date` should be a date, but at least one value is not a valid date.
- `salary` contains dollar signs, commas, blanks, and invalid text.
- `age` contains missing or invalid values.
- `employee_id` 102 appears more than once.
- Some values may be valid technically but suspicious, such as an age of 150.

Today we will focus on structural errors, duplicates, and missing values. We will not fully standardize categories yet.

## Part 3: Create a Safe Working Copy

Never overwrite your raw data immediately. Keep the original DataFrame unchanged and create a working copy.

This is one of the most important habits in data cleaning.

In [8]:
# Keep the raw df unchanged and clean a copy
raw_df = df.copy()
df_clean = df.copy()

# Start a simple cleaning log
cleaning_log = []

print("Raw data shape:", raw_df.shape)
print("Working copy shape:", df_clean.shape)

Raw data shape: (13, 8)
Working copy shape: (13, 8)


## Part 4: Correct Numeric Data Types

The first structural problem is `employee_id`. It should be numeric, but one row contains the text value `error`.

We will use `pd.to_numeric()` with `errors="coerce"`.

When we use `errors="coerce"`, any value that cannot be converted becomes a missing value, shown as `NaN` or `<NA>`. This is useful because it exposes invalid values instead of silently hiding them.

In [9]:
# Convert employee_id to a numeric value.
# Invalid values such as "error" become NaN.
df_clean["employee_id"] = pd.to_numeric(df_clean["employee_id"], errors="coerce")

# Use pandas nullable integer type so missing IDs can exist temporarily
df_clean["employee_id"] = df_clean["employee_id"].astype("Int64")

cleaning_log.append("Converted employee_id to numeric using pd.to_numeric(errors='coerce'). Invalid IDs became missing values.")

df_clean[["employee_id"]]

,employee_id
0,101
1,102
2,103
3,104
4,105
5,106
6,107
7,108
8,102
9,110


### Cleaning Salary Values

The `salary` column is trickier because it may contain values like `$95,000`. If we directly convert those strings to numbers, Pandas may treat them as invalid because of the dollar sign and comma.

So we will first remove `$` and `,`, then convert the result to numeric.

In [10]:
# Show the original salary values before conversion
df_clean["salary"]

0        85000
1        92000
2        71000
3       115000
4        89000
5        95000
6          NaN
7       120000
8        91000
9        68000
10    $91,000 
11       88000
12     5900000
Name: salary, dtype: str

In [11]:
# Remove dollar signs and commas, then convert to numeric
salary_cleaned = (
    df_clean["salary"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_clean["salary"] = pd.to_numeric(salary_cleaned, errors="coerce")

cleaning_log.append("Removed dollar signs and commas from salary, then converted salary to numeric. Invalid salary values became missing values.")

df_clean[["salary"]]

,salary
0,85000
1,92000
2,71000
3,115000
4,89000
5,95000
6,<NA>
7,120000
8,91000
9,68000


### Cleaning Age Values

The `age` column should also be numeric. If values such as `thirty` appear, they should be converted to missing values during this first structural cleaning pass.

In [12]:
df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")

cleaning_log.append("Converted age to numeric using pd.to_numeric(errors='coerce'). Invalid age values became missing values.")

df_clean[["age"]]

,age
0,34.0
1,45.0
2,29.0
3,38.0
4,NaN
5,41.0
6,33.0
7,52.0
8,31.0
9,58.0


## Part 5: Correct Date Data Types

The `hire_date` column should be a date. Because the raw file contains more than one valid date format, we will use `format="mixed"` so Pandas evaluates each value separately. We will also use `errors="coerce"` so truly invalid dates become `NaT`, which means Not a Time. You can think of `NaT` as the date/time version of a missing value.

In [13]:
df_clean["hire_date"] = pd.to_datetime(
    df_clean["hire_date"],
    errors="coerce",
    format="mixed",
)

cleaning_log.append("Converted hire_date to datetime with mixed-format parsing and errors='coerce'. Invalid dates became NaT.")

df_clean[["hire_date"]]

,hire_date
0,2022-04-15
1,2021-08-20
2,2022-01-02
3,2023-11-10
4,2020-05-30
5,NaT
6,2023-02-21
7,2021-09-01
8,2024-01-15
9,2019-07-22


## Part 6: Check What Changed

After type conversion, inspect the data again. Notice how invalid values have become missing values. This is a good thing because now they can be counted and handled consistently.

In [14]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   employee_id  12 non-null     Int64         
 1   first_name   13 non-null     str           
 2   last_name    13 non-null     str           
 3   hire_date    12 non-null     datetime64[us]
 4   department   13 non-null     str           
 5   salary       12 non-null     Int64         
 6   age          12 non-null     float64       
 7   full_time    12 non-null     object        
dtypes: Int64(2), datetime64[us](1), float64(1), object(1), str(3)
memory usage: 990.0+ bytes


In [15]:
df_clean.isna().sum()

employee_id    1
first_name     0
last_name      0
hire_date      1
department     0
salary         1
age            1
full_time      1
dtype: int64

## Part 7: Identify Duplicates

There are two common ways to think about duplicates.

A full-row duplicate means every value in the row is repeated. A key-based duplicate means a specific identifier, such as `employee_id`, appears more than once.

For this employee dataset, `employee_id` should identify one employee. So we care about key-based duplicates.

In [16]:
# Full-row duplicate count
df_clean.duplicated().sum()

np.int64(0)

In [17]:
# Key-based duplicate count for employee_id
df_clean["employee_id"].duplicated().sum()

np.int64(1)

In [18]:
# Show rows with duplicated employee_id values
# keep=False marks all occurrences of the duplicate ID, not just the later ones.
df_clean[df_clean["employee_id"].duplicated(keep=False)].sort_values("employee_id")

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
1,102,Jane,Doe,2021-08-20,Sales,92000,45.0,True
8,102,Emily,White,2024-01-15,Sales,91000,31.0,True


## Part 8: Remove Key-Based Duplicates

Because `employee_id` is supposed to be unique, we will keep the first occurrence and remove later duplicate rows.

In practice, we would not always do this automatically. For example, if the two rows had conflicting salaries or hire dates, we would need to investigate. In this small lab dataset, the repeated employee 102 appears to be a duplicate record.

In [19]:
before_rows = df_clean.shape[0]

df_clean = df_clean.drop_duplicates(subset=["employee_id"], keep="first")

after_rows = df_clean.shape[0]
rows_removed = before_rows - after_rows

cleaning_log.append(f"Removed {rows_removed} duplicate row(s) based on employee_id, keeping the first occurrence.")

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", rows_removed)

Rows before: 13
Rows after: 12
Rows removed: 1


## Part 9: Inspect Missing Data After Type Conversion

Now we need to decide what to do with missing values. Some missing values existed in the raw data. Others were created by `errors="coerce"` when invalid values were converted to missing values.

This is normal. Type conversion often reveals hidden data quality problems.

In [20]:
df_clean.isna().sum()

employee_id    1
first_name     0
last_name      0
hire_date      1
department     0
salary         1
age            1
full_time      1
dtype: int64

In [21]:
# Show rows with any missing values
missing_rows = df_clean[df_clean.isna().any(axis=1)]
missing_rows

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
4,105,Mike,Brown,2020-05-30,sales,89000,NaN,False
5,106,Susan,Davis,NaT,Sales,95000,41.0,True
6,107,Chris,Wilson,2023-02-21,Marketing,<NA>,33.0,True
7,108,Laura,Taylor,2021-09-01,Engineering,120000,52.0,NaN
11,<NA>,Kevin,Clark,2024-02-05,Sales,88000,35.0,True


## Part 10: Handle Missing Critical Fields by Dropping Rows

Some fields are critical. In this dataset, `employee_id` is the identifier, and `hire_date` is important for many HR analyses.

For this lab, we will drop rows with missing `employee_id` or missing `hire_date`.

This is a classroom decision. In practice, we might first go back to the HR system or source database to investigate.

In [22]:
before_rows = df_clean.shape[0]

df_clean = df_clean.dropna(subset=["employee_id", "hire_date"])

after_rows = df_clean.shape[0]
rows_removed = before_rows - after_rows

cleaning_log.append(f"Dropped {rows_removed} row(s) with missing critical employee_id or hire_date.")

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", rows_removed)

Rows before: 12
Rows after: 10
Rows removed: 2


## Part 11: Impute Missing Numeric Values Using the Median

For this lab, we will impute missing `salary` and `age` values using the median.

The median is often safer than the mean when a column has outliers. For example, a single extremely high salary can pull the mean upward, but the median is more resistant to extreme values.

This does not mean median imputation is always best. It is just a simple first strategy.

In [23]:
# Calculate medians before filling missing values
salary_median = df_clean["salary"].median()
age_median = df_clean["age"].median()

print("Salary median:", salary_median)
print("Age median:", age_median)

Salary median: 91000.0
Age median: 38.0


In [24]:
# Fill missing salary and age values with their medians
df_clean["salary"] = df_clean["salary"].fillna(salary_median)
df_clean["age"] = df_clean["age"].fillna(age_median)

cleaning_log.append(f"Imputed missing salary values with the median salary: {salary_median}.")
cleaning_log.append(f"Imputed missing age values with the median age: {age_median}.")

df_clean[["salary", "age"]].head()

,salary,age
0,85000,34.0
1,92000,45.0
2,71000,29.0
3,115000,38.0
4,89000,38.0


## Part 12: Check for Suspicious Values

Cleaning missing values does not mean the data is perfect. We also need to look for suspicious values.

For example, the age value 150 is technically numeric, but it is probably not realistic for an employee. We will flag it here, but we will not automatically fix it in this lab.

In [25]:
# Show unusually high ages
suspicious_age = df_clean[df_clean["age"] > 100]
suspicious_age

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
12,113,Brian,Hall,2022-06-18,Engineering,5900000,999.0,True


In [26]:
if not suspicious_age.empty:
    cleaning_log.append("Flagged age values above 100 for further review. These were not automatically changed in this lab.")

## Part 13: Verify the Cleaned Dataset

Now inspect the cleaned DataFrame. We want to confirm that the major structural issues were addressed.

In [27]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 10 entries, 0 to 12
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   employee_id  10 non-null     Int64         
 1   first_name   10 non-null     str           
 2   last_name    10 non-null     str           
 3   hire_date    10 non-null     datetime64[us]
 4   department   10 non-null     str           
 5   salary       10 non-null     Int64         
 6   age          10 non-null     float64       
 7   full_time    9 non-null      object        
dtypes: Int64(2), datetime64[us](1), float64(1), object(1), str(3)
memory usage: 740.0+ bytes


In [28]:
df_clean.isna().sum()

employee_id    0
first_name     0
last_name      0
hire_date      0
department     0
salary         0
age            0
full_time      1
dtype: int64

In [29]:
df_clean.shape

(10, 8)

In [30]:
df_clean.head()

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
0,101,John,Smith,2022-04-15,Marketing,85000,34.0,True
1,102,Jane,Doe,2021-08-20,Sales,92000,45.0,True
2,103,Peter,Jones,2022-01-02,HR,71000,29.0,True
3,104,Mary,Johnson,2023-11-10,Engineering,115000,38.0,True
4,105,Mike,Brown,2020-05-30,sales,89000,38.0,False


## Part 14: Review the Cleaning Log

A cleaning log helps you remember what changed and why. It is also useful for project reports, reproducibility, and professional communication.

In [31]:
for i, item in enumerate(cleaning_log, start=1):
    print(f"{i}. {item}")

1. Converted employee_id to numeric using pd.to_numeric(errors='coerce'). Invalid IDs became missing values.
2. Removed dollar signs and commas from salary, then converted salary to numeric. Invalid salary values became missing values.
3. Converted age to numeric using pd.to_numeric(errors='coerce'). Invalid age values became missing values.
4. Converted hire_date to datetime with mixed-format parsing and errors='coerce'. Invalid dates became NaT.
5. Removed 1 duplicate row(s) based on employee_id, keeping the first occurrence.
6. Dropped 2 row(s) with missing critical employee_id or hire_date.
7. Imputed missing salary values with the median salary: 91000.0.
8. Imputed missing age values with the median age: 38.0.
9. Flagged age values above 100 for further review. These were not automatically changed in this lab.


## Part 15: Save the Cleaned Data

For this lab, we will save the cleaned dataset as a new CSV file in the `data/processed` folder. We are not overwriting the protected raw file. Keeping raw inputs and generated outputs separate makes the workflow easier to understand and reproduce.

In [32]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "employee_data_cleaned_week3.csv"
df_clean.to_csv(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")

Cleaned data saved to: ../data/processed/employee_data_cleaned_week3.csv


## Part 16: Your Turn

Answer the following questions in the markdown cell below.

1. Which cleaning step created new missing values? Why did that happen?
2. Why did we use median imputation for `salary` and `age` instead of mean imputation?
3. Why is it safer to use `drop_duplicates(subset=["employee_id"])` instead of only `drop_duplicates()` in this dataset?
4. What issue did we flag but not automatically fix?
5. What is one cleaning decision in this lab that might require more investigation in a real organization?

### Your Answers

Write your answers here.

1. In part 4, correcting numeric data types created new missing values because we used the function error="coerce" using Pandas. 
2. We used median imputation instead of mean imputation because it is often safer when a column has outliers. 
3. Because it is a small dataset and it is a repeated employee. 
4. The age, as there is one that is showing as 150 which is not realistic for an employee. 
5. Dropping the rows with missing employee_id or missing hire_date, but they would require us to go to the HR system or source database to investigate in a real organization. 

## Part 17: Offline Assignment - Independent Output Audit

Complete this section independently after the guided lab. Do not change the raw CSV file. Your goal is to decide whether the saved cleaned file is ready for later analysis and to support that decision with evidence.

1. Reload the CSV from `output_path` into a new DataFrame named `offline_check`.
2. Display its first rows and inspect its structure.
3. Complete and display the validation summary started below. It checks total rows, unique employee IDs, duplicate IDs, missing critical values, missing numeric values, and ages above 100.
4. Use the results to write a short audit response. Explain what passed, what concern remains, and what business rule or source-system question is needed before changing a suspicious value.

Do not silently fix the age above 100. Module 3 emphasizes that a suspicious value should be investigated rather than guessed.

In [33]:
# Offline Assignment
offline_check = pd.read_csv(output_path)
offline_check.head()

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
0,101,John,Smith,2022-04-15,Marketing,85000,34.0,True
1,102,Jane,Doe,2021-08-20,Sales,92000,45.0,True
2,103,Peter,Jones,2022-01-02,HR,71000,29.0,True
3,104,Mary,Johnson,2023-11-10,Engineering,115000,38.0,True
4,105,Mike,Brown,2020-05-30,sales,89000,38.0,False


In [35]:
# Offline Assignment - Complete the four TODO values.
# Hints:
# - Use .duplicated().sum() on employee_id for duplicate IDs.
# - Select the named columns first, then use .isna().sum().sum() for missing totals.
# - Use a Boolean condition on age, then sum the True values.
offline_validation_summary = pd.Series({
    "total_rows": len(offline_check),
    "unique_nonmissing_employee_ids": offline_check["employee_id"].dropna().nunique(),
    "duplicate_employee_ids": offline_check["employee_id"].duplicated().sum(),
    "missing_critical_values": offline_check[["employee_id", "hire_date"]].isna().sum().sum(),
    "missing_numeric_values": offline_check[["salary", "age"]].isna().sum().sum(),
    "ages_above_100": (offline_check["age"] > 100).sum()
}, name="result")

offline_validation_summary


total_rows                        10
unique_nonmissing_employee_ids    10
duplicate_employee_ids             0
missing_critical_values            0
missing_numeric_values             0
ages_above_100                     1
Name: result, dtype: int64

### Offline Assignment Audit Response

The validation results showed that the dataset has 10 total rows and 10 unique employee IDs, with no duplicate employee IDs. There are also no missing critical values or missing numeric values, which means the dataset is mostly ready for later analysis. The only thing that is not good is that one age is above 100, which would be the only thing that would need to be investigated before doing the analysis. To investigate it, I would check the HR system or the source database to see if the age was entered incorrectly in error. If it was a data entry error, I would correct it if the correct age if it is available or mark it as missing until it can be verified with HR. 

## Submission Reminder

Before submitting:

1. Run all cells from top to bottom.
2. Make sure the notebook has no errors.
3. Complete the guided reflection and offline assignment.
4. Confirm that `data/processed/employee_data_cleaned_week3.csv` exists.
5. Save the notebook.
6. Commit and push your work to GitHub.

Suggested Git commands:

```bash
git add .
git commit -m "Completed Week 3 data cleaning lab"
git push
```

## Optional Practice: Clean a New Mini Dataset (Not Graded)

This short practice does **not** count toward your lab grade. Try it after completing the required work if you want another repetition of the Week 3 cleaning workflow.

The mini dataset below contains the same kinds of issues as the guided lab, but the values are different. Work on a copy, keep the raw practice data unchanged, and use the self-check only after you try the task yourself.


In [ ]:
practice_raw = pd.DataFrame({
    "employee_id": ["201", "202", "202", "bad", "205"],
    "salary": ["$52,000", "61,000", "$61,000", "not available", ""],
    "hire_date": ["2024-01-15", "02/01/2024", "02/01/2024", "bad date", "2024-03-10"],
    "age": ["29", "unknown", "35", "41", "150"],
    "department": ["Sales", "Finance", "Finance", "IT", "Operations"],
})

practice_raw


### Practice Task

Create a DataFrame named `practice_clean` and complete this sequence:

1. Copy `practice_raw` instead of changing it directly.
2. Convert `employee_id` to a nullable integer and convert invalid IDs to missing values.
3. Remove dollar signs and commas from `salary`, then convert it to numeric.
4. Convert `age` to numeric and `hire_date` to datetime; coerce invalid values.
5. Remove duplicate employee IDs, keeping the first record.
6. Drop rows missing the critical `employee_id` or `hire_date` fields.
7. Fill missing salary and age values with the median of the remaining records.

Hints:

- Reuse the `pd.to_numeric(..., errors="coerce")`, string replacement, and `pd.to_datetime(..., errors="coerce", format="mixed")` patterns from the guided lab.
- Use `drop_duplicates(subset=["employee_id"], keep="first")` before `dropna()`.
- Calculate each median after the duplicate and critical-field steps, then use `fillna()`.


In [ ]:
# Optional practice workspace
# Build a cleaned DataFrame named practice_clean.
#
# practice_clean = practice_raw.copy()
# Continue the seven steps here.


In [ ]:
# Run this self-check after you create practice_clean.
if "practice_clean" not in globals():
    print("Complete the optional practice first, then rerun this self-check.")
else:
    practice_check = pd.Series({
        "raw_rows_unchanged": len(practice_raw) == 5,
        "clean_rows": len(practice_clean),
        "duplicate_employee_ids": practice_clean["employee_id"].duplicated().sum(),
        "missing_salary_or_age": practice_clean[["salary", "age"]].isna().sum().sum(),
        "ages_above_100": (practice_clean["age"] > 100).sum(),
    }, name="result")
    display(practice_check)
    print("Expected: 3 clean rows, 0 duplicate IDs, 0 missing salary/age values, and 1 age above 100.")


### Think About the Result

Passing the structural checks does not prove every value is valid. The remaining age above 100 should still be flagged for investigation rather than silently replaced. Compare your steps with the main lab and identify which checks address structure, which address missingness, and which require a business rule.
